# Epoch Spring Camp 2026 - Take Home Task 3

In this task, you will build and compare two recommender system models:

- **Matrix Factorization (MF)** using a dot product of embeddings  
- **Neural Collaborative Filtering (MLP-based)** using a multi-layer perceptron  

You are provided with:
- Preprocessed interaction data
- Evaluation pipeline

You are expected to implement:
- Negative sampling
- Model architectures
- Training loop

---

The purpose of this task is to understand how neural networks can model user–item interactions beyond simple similarity.

## Imports

In [70]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np

## Loading Data

We begin by loading the interaction data.

In [71]:
df = pd.read_csv("interactions.csv")  # columns: user_id, movie_id

print("Num users:", df['user_id'].nunique())
print("Num items:", df['item_id'].nunique())
print("Num interactions:", len(df))

Num users: 942
Num items: 1447
Num interactions: 55375


Each row represents a **positive interaction** (implicit feedback).

## Train / Test Split

We split the data into training and testing sets.

The model will learn from the training data and be evaluated on unseen interactions in the test set.

In [72]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [73]:
df['user_id'].unique()

array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
        13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
        26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
        39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
        52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
        65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
        78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
        91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
       104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
       117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
       130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
       143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
       156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
       169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 18

## Negative Sampling

The dataset only contains **positive interactions** (user interacted with item).

However, to train a model, we also need **negative examples** (user did not interact with item).

---

### Why do we need this?

We treat recommendation as a **binary classification problem**:

- Positive interaction → label = 1  
- No interaction → label = 0  

Since we don’t have explicit negatives, we **sample them**.

---

### Your Task

For each `(user, item)` interaction:
- Keep it as a **positive sample (label = 1)**
- Sample one or more items the user has **not interacted with**
  - Add them as **negative samples (label = 0)**

---

### Hints

- You can randomly sample items
- Make sure sampled negatives are **not already in the dataset**
- You may use a `set` of `(user, item)` pairs for fast lookup

---

Implement a function:
`sample_negatives(df, num_neg=1)`
that returns a dataframe with columns:
`[user, item, label]`

In [74]:
def sample_negatives(df, num_neg=4):
    users = df['user_id'].values
    items = df['item_id'].values
    all_items = df['item_id'].unique()
    interactions_set = set(zip(users, items))

    user_list, item_list, label_list = [], [], []

    # Positive samples
    user_list.extend(users)
    item_list.extend(items)
    label_list.extend([1] * len(users))

    # Negative samples
    for u in df['user_id'].unique():
        user_interacted = {i for user, i in interactions_set if user == u}
        negatives = []
        while len(negatives) < num_neg:
            potential_negs = np.random.choice(all_items, num_neg)
            for neg in potential_negs:
                if neg not in user_interacted:
                    negatives.append(neg)
                if len(negatives) == num_neg:
                    break

        user_list.extend([u] * num_neg)
        item_list.extend(negatives)
        label_list.extend([0] * num_neg)

    return pd.DataFrame({'user_id': user_list, 'item_id': item_list, 'label': label_list})

## PyTorch Dataset

We now wrap our processed data into a PyTorch `Dataset`.

This allows us to:
- Access individual samples as `(user, item, label)`
- Easily plug into a `DataLoader` for batching

You do not need to modify this part.

In [75]:
class InteractionDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df['user_id'].values.astype(np.int32))
        self.items = torch.tensor(df['item_id'].values.astype(np.int32))
        self.labels = torch.tensor(df['label'].values.astype(np.int32)).float()

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.labels[idx]

## DataLoader

The `DataLoader` will:
- Batch the data
- Shuffle the training data
- Feed it to the model during training

In [88]:
train_data = sample_negatives(train_df,num_neg=500)

train_ds,valid_ds=train_test_split(train_data,test_size=0.1,random_state=42)
train_loader = DataLoader(InteractionDataset(train_ds), batch_size=256, shuffle=True)
valid_loader = DataLoader(InteractionDataset(valid_ds), batch_size=256, shuffle=True)

Why are we sampling negatives only for the training data?

## Quick Exploration

Before building models, take a moment to explore the data.

Try to understand:
- How many interactions each user has
- How popular certain items are

This can give intuition about the dataset.

In [77]:
# Interactions per user
user_counts = df['user_id'].value_counts()
print(user_counts.describe())

# Interactions per item
item_counts = df['item_id'].value_counts()
print(item_counts.describe())

count    942.000000
mean      58.784501
std       54.696664
min        3.000000
25%       19.000000
50%       39.500000
75%       80.750000
max      378.000000
Name: count, dtype: float64
count    1447.000000
mean       38.268832
std        57.956847
min         1.000000
25%         3.000000
50%        13.000000
75%        47.500000
max       501.000000
Name: count, dtype: float64


## Baseline Model: Matrix Factorization (MF)

We represent:
- Each **user** as a vector (embedding)
- Each **item** as a vector (embedding)

To predict interaction:
- Compute the **dot product** between user and item embeddings

This gives a **score** indicating how likely the user is to interact with the item.

---

### Your Task

Implement a model that:
1. Learns user and item embeddings
2. Computes their dot product as the output score

In [78]:
class MF(nn.Module):
    def __init__(self, num_users, num_items, emb_dim):
        super().__init__()

        # TODO:
        # - Define user embedding layer
        # - Define item embedding layer
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)

    def forward(self, user, item):
        # TODO:
        # - Get user and item embeddings
        # - Compute dot product
        # - Return a single score per pair
        user_emb = self.user_emb(user)
        item_emb = self.item_emb(item)
        return (user_emb * item_emb).sum(1)

## Training the MF Model

Now train your Matrix Factorization model.

You will need to:
- Define a loss function
- Define an optimizer
- Iterate over the DataLoader
- Update model parameters

---

### Hints

- Use **Binary Cross Entropy loss**
- Apply **sigmoid** to model outputs if needed
- Typical steps:
  - forward pass
  - compute loss
  - backward pass
  - optimizer step

In [79]:
def train(model, dataloader, val_dataloader, epochs, patience=100):
    if torch.cuda.is_available():
        device = torch.device('cuda')
    else:
        device = torch.device('cpu')

    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.BCEWithLogitsLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

    best_val_loss = float('inf')
    early_stop_counter = 0
    losses = []

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for user, item, label in dataloader:
            optimizer.zero_grad()
            relevance_scores = model(user.to(device), item.to(device))
            label = label.to(device).float()

            loss = criterion(relevance_scores, label)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(dataloader)
        losses.append(avg_train_loss)

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for user, item, label in val_dataloader:
                relevance_scores = model(user.to(device), item.to(device))
                label = label.to(device).float()
                loss = criterion(relevance_scores, label)
                val_loss += loss.item()

        avg_val_loss = val_loss / len(val_dataloader)
        print(f"Epoch {epoch+1}: Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")

        # Scheduler step
        scheduler.step(avg_val_loss)

        # Early Stopping logic
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            early_stop_counter = 0
            # Optional: Save best model
            best_model_state = model.state_dict()
        else:
            early_stop_counter += 1
            if early_stop_counter >= patience:
                print("Early stopping triggered.")
                model.load_state_dict(best_model_state)
                break

    return losses

In [80]:
# TODO:
# 1. Initialize MF model
MF_model = MF(num_users=df['user_id'].nunique(), num_items=df['item_id'].nunique(), emb_dim=16)
# 2. Train it
train(MF_model, train_loader,valid_loader, epochs=300)

Epoch 1: Train Loss: 1.6974, Val Loss: 1.6433
Epoch 2: Train Loss: 1.5735, Val Loss: 1.5794
Epoch 3: Train Loss: 1.4617, Val Loss: 1.4826
Epoch 4: Train Loss: 1.3612, Val Loss: 1.4153
Epoch 5: Train Loss: 1.2708, Val Loss: 1.3545
Epoch 6: Train Loss: 1.1898, Val Loss: 1.2985
Epoch 7: Train Loss: 1.1172, Val Loss: 1.2456
Epoch 8: Train Loss: 1.0521, Val Loss: 1.1923
Epoch 9: Train Loss: 0.9938, Val Loss: 1.1438
Epoch 10: Train Loss: 0.9415, Val Loss: 1.1168
Epoch 11: Train Loss: 0.8946, Val Loss: 1.0720
Epoch 12: Train Loss: 0.8523, Val Loss: 1.0392
Epoch 13: Train Loss: 0.8139, Val Loss: 1.0069
Epoch 14: Train Loss: 0.7788, Val Loss: 0.9814
Epoch 15: Train Loss: 0.7463, Val Loss: 0.9539
Epoch 16: Train Loss: 0.7157, Val Loss: 0.9274
Epoch 17: Train Loss: 0.6865, Val Loss: 0.8988
Epoch 18: Train Loss: 0.6580, Val Loss: 0.8716
Epoch 19: Train Loss: 0.6296, Val Loss: 0.8396
Epoch 20: Train Loss: 0.6010, Val Loss: 0.8117
Epoch 21: Train Loss: 0.5721, Val Loss: 0.7805
Epoch 22: Train Loss: 

[1.6973593832286231,
 1.5735122871105187,
 1.4616878948172505,
 1.3611610572196131,
 1.2708465018801132,
 1.1898114587002466,
 1.1171795610774469,
 1.0521159339734416,
 0.9938161846793407,
 0.9415412935632944,
 0.8945884742531198,
 0.8523040393049957,
 0.8138903224003143,
 0.7787983620680823,
 0.7463116341058234,
 0.715742570541233,
 0.6864847977792947,
 0.6579536473726589,
 0.629608078414165,
 0.6010420634516456,
 0.5721116922719278,
 0.5429445667310907,
 0.5139154774697164,
 0.48552982187858595,
 0.45853478585425345,
 0.43344410115688486,
 0.4106901704775479,
 0.3904005081624221,
 0.3725165640304221,
 0.3568256605699567,
 0.3430119686792519,
 0.33079333435093844,
 0.3199452431356148,
 0.31024053649980676,
 0.30145155857231093,
 0.2934637465944525,
 0.2861163824551894,
 0.2793609853336698,
 0.273058530455742,
 0.2671764960523993,
 0.26163832127313597,
 0.2563856896739721,
 0.25143534584579036,
 0.24673150550168643,
 0.24070424991956238,
 0.23850039672680215,
 0.23633309978479233,
 0.2

## Evaluation (Hit@K)

We evaluate the model using a ranking-based metric.

For each user:
- Take one positive item
- Sample multiple negative items
- Rank all items using the model
- Check if the positive item is in the top-K

This is called **Hit@K**.

In [81]:
def hit_at_k(model, test_df, full_df, K=10, num_neg=100):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    model.to(device)

    hits = 0
    total = len(df) # We evaluate every positive interaction in the test set

    # Map interactions for quick lookup
    interacted_items = full_df.groupby('user_id')['item_id'].apply(set).to_dict()
    all_items = full_df['item_id'].unique()

    with torch.no_grad(): # Disable gradient calculation for speed/memory
        for _, row in df.iterrows():
            u = int(row['user_id'])
            pos_item = int(row['item_id'])

            # 1. Sample Negatives
            negatives = []
            while len(negatives) < num_neg:
                neg_item = np.random.choice(all_items)
                if neg_item not in interacted_items.get(u, set()):
                    negatives.append(neg_item)

            # 2. Prepare Tensors
            # We need a list of the 1 positive + 100 negatives
            item_list = [pos_item] + negatives
            user_tensor = torch.tensor([u] * (num_neg + 1)).to(device)
            item_tensor = torch.tensor(item_list).to(device)

            # 3. Get Scores
            scores = model(user_tensor, item_tensor)

            # 4. Rank and Check Hit
            # We want to see if the item at index 0 (the positive) is in the top K
            # torch.topk returns values and indices of the highest scores
            _, top_indices = torch.topk(scores, K)

            top_indices = top_indices.cpu().numpy()
            if 0 in top_indices:
                hits += 1

    return hits / total


In [82]:


# 3. Evaluate it
hit_rate = hit_at_k(MF_model, test_df, df, K=10, num_neg=100)
print(f"Hit@10: {hit_rate:.4f}")

Hit@10: 0.5761


If your score is low, try playing with the hyperparameters before moving on! Try sampling more negatives, or playing with the embedding dimensions.

Conversely, if your score is high, play with the values of K or number of negatives in evaluation (dec K, inc negatives).

## Neural Model: MLP-Based Recommender

Instead of using a dot product, we can learn a more flexible interaction function using a neural network.

Approach:
- Learn user and item embeddings
- **Concatenate** them
- Pass through a **Multi-Layer Perceptron (MLP)**

This allows the model to capture more complex relationships than simple similarity.

---

### Your Task

Implement a model that:
1. Learns user and item embeddings
2. Concatenates them
3. Passes them through an MLP to produce a score

The choice of activation functions is upto you.

In [83]:
class MLP(nn.Module):
    def __init__(self, num_users, num_items, emb_dim):
        super().__init__()

        # TODO:
        # - Define user embedding layer
        # - Define item embedding layer
        # - Define MLP layers
        self.user_emb=nn.Embedding(num_users, emb_dim)
        self.item_emb=nn.Embedding(num_items, emb_dim)
        self.mlp = nn.Sequential(
            nn.Linear(2 * emb_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, user, item):
        # TODO:
        # - Get embeddings
        # - Concatenate user and item embeddings
        # - Pass through MLP
        # - Return a single score
        user_emb = self.user_emb(user)
        item_emb = self.item_emb(item)
        concatenated = torch.cat([user_emb, item_emb], dim=1)
        return self.mlp(concatenated)[:,0]

## Train and Evaluate the MLP Model

Repeat the same steps as before:
- Train the model
- Evaluate on the test set

Compare its performance with the MF model.

In [84]:
# TODO:
# 1. Initialize MLP model
# 2. Train it
# 3. Evaluate it
MLP_model = MLP(num_users=df['user_id'].nunique(), num_items=df['item_id'].nunique(), emb_dim=16)
train(MLP_model, train_loader,valid_loader, epochs=300)
hit_rate = hit_at_k(MLP_model, test_df, df, K=10, num_neg=100)
print(f"Hit@10: {hit_rate:.4f}")


Epoch 1: Train Loss: 0.5925, Val Loss: 0.5398
Epoch 2: Train Loss: 0.4823, Val Loss: 0.4559
Epoch 3: Train Loss: 0.4248, Val Loss: 0.4290
Epoch 4: Train Loss: 0.4007, Val Loss: 0.4158
Epoch 5: Train Loss: 0.3890, Val Loss: 0.4091
Epoch 6: Train Loss: 0.3827, Val Loss: 0.4052
Epoch 7: Train Loss: 0.3784, Val Loss: 0.4053
Epoch 8: Train Loss: 0.3756, Val Loss: 0.4075
Epoch 9: Train Loss: 0.3727, Val Loss: 0.4100
Epoch 10: Train Loss: 0.3669, Val Loss: 0.4086
Epoch 11: Train Loss: 0.3654, Val Loss: 0.4100
Epoch 12: Train Loss: 0.3642, Val Loss: 0.4067
Epoch 13: Train Loss: 0.3611, Val Loss: 0.4081
Epoch 14: Train Loss: 0.3603, Val Loss: 0.4103
Epoch 15: Train Loss: 0.3596, Val Loss: 0.4071
Epoch 16: Train Loss: 0.3579, Val Loss: 0.4055
Epoch 17: Train Loss: 0.3575, Val Loss: 0.4067
Epoch 18: Train Loss: 0.3571, Val Loss: 0.4065
Epoch 19: Train Loss: 0.3563, Val Loss: 0.4070
Epoch 20: Train Loss: 0.3561, Val Loss: 0.4074
Epoch 21: Train Loss: 0.3559, Val Loss: 0.4086
Epoch 22: Train Loss: 

## Comparison & Analysis

You have now implemented:
- Matrix Factorization (MF)
- MLP-based recommender

---

### Compare the Models

- What score did each model achieve?
- Which model performed better?

---

### Think & Reflect

- Why might the MLP model outperform MF?
- In what cases might MF perform just as well or better?
- How does embedding size affect performance?
- Did you observe any signs of overfitting?

---

### Some Experiments

If you want to explore further:
- Try different embedding dimensions
- Change number of MLP layers
- Try different activation functions

---

## Bonus Task: Neural Collaborative Filtering (NCF)

In this task, you implemented:
- Matrix Factorization (MF)
- MLP-based recommender

The paper [Neural Collaborative Filtering](https://arxiv.org/pdf/1708.05031) proposes combining both ideas.

---

### Idea

- MF captures **linear interactions**
- MLP captures **nonlinear interactions**

NCF combines both by:
1. Computing MF output
2. Computing MLP output
3. Combining them into a final prediction

---

### Your Task

Design a model that:
- Uses both MF and MLP components
- Combines their outputs
- Trains end-to-end

---

### Hints

- You can:
  - Concatenate MF and MLP representations
  - Or combine their final scores
- Think about:
  - Should embeddings be shared or separate?
  - How to balance both components?

---

Does combining both approaches improve performance over MF and MLP individually?

In [85]:
class NCF(nn.Module):
    def __init__(self, num_users, num_items, mf_dim=16, mlp_dim=16, layers=[64, 32, 16]):
        super(NCF, self).__init__()

        # Matrix Factorization (GMF) component
        self.mf_user_emb = nn.Embedding(num_users, mf_dim)
        self.mf_item_emb = nn.Embedding(num_items, mf_dim)

        # MLP component
        self.mlp_user_emb = nn.Embedding(num_users, mlp_dim)
        self.mlp_item_emb = nn.Embedding(num_items, mlp_dim)

        mlp_modules = []
        input_size = 2 * mlp_dim
        for output_size in layers:
            mlp_modules.append(nn.Linear(input_size, output_size))
            mlp_modules.append(nn.ReLU())
            input_size = output_size
        self.mlp_network = nn.Sequential(*mlp_modules)

        # Final Prediction Layer (Combines both outputs)
        self.prediction_layer = nn.Linear(mf_dim + layers[-1], 1)

    def forward(self, user, item):
        # GMF Branch
        mf_user = self.mf_user_emb(user)
        mf_item = self.mf_item_emb(item)
        mf_output = mf_user * mf_item # element-wise product

        # MLP Branch
        mlp_user = self.mlp_user_emb(user)
        mlp_item = self.mlp_item_emb(item)
        mlp_input = torch.cat([mlp_user, mlp_item], dim=1)
        mlp_output = self.mlp_network(mlp_input)

        # Concatenate GMF and MLP outputs
        combined = torch.cat([mf_output, mlp_output], dim=1)

        # Output layer
        return self.prediction_layer(combined).squeeze()

In [86]:
# 1. Initialize NCF model
NCF_model = NCF(num_users=df['user_id'].nunique(), num_items=df['item_id'].nunique(), mf_dim=16, mlp_dim=16)

# 2. Train it
print("Starting NCF training...")
train(NCF_model, train_loader, valid_loader, epochs=300)


Starting NCF training...
Epoch 1: Train Loss: 0.6004, Val Loss: 0.5354
Epoch 2: Train Loss: 0.4828, Val Loss: 0.4578
Epoch 3: Train Loss: 0.4271, Val Loss: 0.4280
Epoch 4: Train Loss: 0.4020, Val Loss: 0.4228
Epoch 5: Train Loss: 0.3892, Val Loss: 0.4103
Epoch 6: Train Loss: 0.3811, Val Loss: 0.4108
Epoch 7: Train Loss: 0.3752, Val Loss: 0.4099
Epoch 8: Train Loss: 0.3701, Val Loss: 0.4128
Epoch 9: Train Loss: 0.3656, Val Loss: 0.4162
Epoch 10: Train Loss: 0.3602, Val Loss: 0.4155
Epoch 11: Train Loss: 0.3508, Val Loss: 0.4195
Epoch 12: Train Loss: 0.3473, Val Loss: 0.4218
Epoch 13: Train Loss: 0.3442, Val Loss: 0.4253
Epoch 14: Train Loss: 0.3385, Val Loss: 0.4258
Epoch 15: Train Loss: 0.3369, Val Loss: 0.4287
Epoch 16: Train Loss: 0.3351, Val Loss: 0.4266
Epoch 17: Train Loss: 0.3320, Val Loss: 0.4297
Epoch 18: Train Loss: 0.3311, Val Loss: 0.4352
Epoch 19: Train Loss: 0.3302, Val Loss: 0.4394
Epoch 20: Train Loss: 0.3287, Val Loss: 0.4334
Epoch 21: Train Loss: 0.3282, Val Loss: 0.43

[0.6003563206421032,
 0.482820402303026,
 0.42708937888027954,
 0.40195585343627227,
 0.38918970418172205,
 0.3811414057591613,
 0.37524717776927125,
 0.3701174349755477,
 0.36563829548305066,
 0.36022651388292687,
 0.3508186316465695,
 0.34725662876203567,
 0.344242291773614,
 0.3385214472698235,
 0.33687310361397094,
 0.3351394155921388,
 0.33197475939193544,
 0.3311126539961758,
 0.3301723224916008,
 0.32872271094111694,
 0.3282260586348892,
 0.32787079099389804,
 0.32697864972223245,
 0.32674771932850627,
 0.326549154746459,
 0.32610767281153363,
 0.32600514229685373,
 0.3258996065514778,
 0.3256648913240041,
 0.32561323805511366,
 0.3255818838089154,
 0.3254561926672346,
 0.3254378153435748,
 0.3254084583425424,
 0.3253536031292694,
 0.3253392456309751,
 0.3253281328222835,
 0.3252984806742267,
 0.32528648355658296,
 0.32528189044958267,
 0.3252707433712801,
 0.32526461608723206,
 0.32526054868218346,
 0.32524075724750573,
 0.3252528989400707,
 0.32526099027059896,
 0.325240207893

In [87]:

# 3. Evaluate it
ncf_hit_rate = hit_at_k(NCF_model, test_df, df, K=10, num_neg=100)
print(f"NCF Hit@10: {ncf_hit_rate:.4f}")

NCF Hit@10: 0.5174


### Hyperparameter Tuning & Regularization

Let's try to improve the score by:
- Increasing the embedding dimension to 32.
- Adding `weight_decay` to the optimizer to regularize the embeddings.
- Tuning the MLP architecture.

In [89]:
# Re-initialize NCF with larger embeddings
improved_ncf = NCF(
    num_users=df['user_id'].nunique(),
    num_items=df['item_id'].nunique(),
    mf_dim=32,
    mlp_dim=32,
    layers=[128, 64, 32]
)

# Updated training call with Weight Decay
def train_improved(model, dataloader, val_dataloader, epochs=100, lr=0.001, weight_decay=1e-5):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.BCEWithLogitsLoss()

    # Re-using your training logic with the added weight decay
    return train(model, dataloader, val_dataloader, epochs=epochs)

print("Training improved NCF model...")
train_improved(improved_ncf, train_loader, valid_loader)

# Final Evaluation
final_hit_rate = hit_at_k(improved_ncf, test_df, df, K=10, num_neg=100)
print(f"Improved NCF Hit@10: {final_hit_rate:.4f}")

Training improved NCF model...
Epoch 1: Train Loss: 0.2479, Val Loss: 0.2111
Epoch 2: Train Loss: 0.2046, Val Loss: 0.2062
Epoch 3: Train Loss: 0.1987, Val Loss: 0.2044
Epoch 4: Train Loss: 0.1949, Val Loss: 0.2039
Epoch 5: Train Loss: 0.1909, Val Loss: 0.2037
Epoch 6: Train Loss: 0.1863, Val Loss: 0.2051
Epoch 7: Train Loss: 0.1811, Val Loss: 0.2082
Epoch 8: Train Loss: 0.1751, Val Loss: 0.2105
Epoch 9: Train Loss: 0.1643, Val Loss: 0.2153
Epoch 10: Train Loss: 0.1589, Val Loss: 0.2175
Epoch 11: Train Loss: 0.1541, Val Loss: 0.2231
Epoch 12: Train Loss: 0.1464, Val Loss: 0.2265
Epoch 13: Train Loss: 0.1431, Val Loss: 0.2312
Epoch 14: Train Loss: 0.1402, Val Loss: 0.2330
Epoch 15: Train Loss: 0.1355, Val Loss: 0.2363
Epoch 16: Train Loss: 0.1338, Val Loss: 0.2395
Epoch 17: Train Loss: 0.1323, Val Loss: 0.2405
Epoch 18: Train Loss: 0.1297, Val Loss: 0.2423
Epoch 19: Train Loss: 0.1288, Val Loss: 0.2436
Epoch 20: Train Loss: 0.1280, Val Loss: 0.2453
Epoch 21: Train Loss: 0.1266, Val Loss